In [1]:
%%capture
! pip install --upgrade kagglehub

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kacpergregorowicz/house-plant-species")

print("Path to dataset files:", path)

100%|██████████| 4.85G/4.85G [01:02<00:00, 82.7MB/s]


Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/kacpergregorowicz/house-plant-species/versions/4


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
import os
import datetime
import matplotlib.pyplot as plt
import cv2

In [4]:
%load_ext tensorboard
!rm -rf ./logs/

In [5]:
os.listdir(path)

['house_plant_species']

In [6]:
PATH_IMAGES = os.path.join(path, "house_plant_species")

print("Number of folders in PATH:", len(os.listdir(PATH_IMAGES)))
print("Number of images in PATH:", sum([len(os.listdir(os.path.join(PATH_IMAGES, folder))) for folder in os.listdir(PATH_IMAGES)]))

Number of folders in PATH: 47
Number of images in PATH: 14790


In [11]:
for folder in os.listdir(PATH_IMAGES):
    plt.figure(figsize=(10, 10))
    plt.suptitle(folder, fontsize=20)
    for i in range(3):
            image = cv2.imread(os.path.join(PATH_IMAGES, folder, os.listdir(os.path.join(PATH_IMAGES, folder))[i]))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            ax = plt.subplot(1, 3, i + 1)
            plt.imshow(image)
            plt.axis('off')
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
#@title Hyperparameters
BATCH_SIZE = 32 # @param ["16","32","64","128"] {"type":"raw"}
IMG_SIZE = 224  # @param ["160","180","224"] {"type":"raw"}
SEED = 42
EPOCHS = 15 # @param ["15","30","50", "100"] {"type":"raw"}

In [ ]:
train_gen = ImageDataGenerator(rescale=1./255,
                               rotation_range=40,
                               width_shift_range=0.2,
                               height_shift_range=0.2,
                               shear_range=0.2,
                               zoom_range=0.2,
                               horizontal_flip=True,
                               fill_mode='nearest')

train_ds = train_gen.flow_from_directory(PATH_IMAGES,
                                         class_mode="categorical",
                                         shuffle=True,
                                         color_mode="rgb",
                                         save_format="jpg",
                                         target_size=(IMG_SIZE, IMG_SIZE),
                                         batch_size=BATCH_SIZE,
                                         subset="training",
                                         seed=SEED)

val_gen = ImageDataGenerator(rescale=1./255,
                             validation_split=0.2)

val_ds = val_gen.flow_from_directory(PATH_IMAGES,
                                    class_mode="categorical",
                                    color_mode="rgb",
                                    save_format="jpg",
                                     target_size=(IMG_SIZE, IMG_SIZE),
                                     batch_size=BATCH_SIZE,
                                     subset="validation",
                                     shuffle=False,
                                     seed=SEED)

Found 11715 images belonging to 47 classes.
Found 2905 images belonging to 47 classes.


In [ ]:
class_names = train_ds.class_indices.keys()
num_classes = len(class_names)

print("Number of classes:", num_classes)
print(class_names)

Number of classes: 47
dict_keys(['African Violet (Saintpaulia ionantha)', 'Aloe Vera', 'Anthurium (Anthurium andraeanum)', 'Areca Palm (Dypsis lutescens)', 'Asparagus Fern (Asparagus setaceus)', 'Begonia (Begonia spp.)', 'Bird of Paradise (Strelitzia reginae)', 'Birds Nest Fern (Asplenium nidus)', 'Boston Fern (Nephrolepis exaltata)', 'Calathea', 'Cast Iron Plant (Aspidistra elatior)', 'Chinese Money Plant (Pilea peperomioides)', 'Chinese evergreen (Aglaonema)', 'Christmas Cactus (Schlumbergera bridgesii)', 'Chrysanthemum', 'Ctenanthe', 'Daffodils (Narcissus spp.)', 'Dracaena', 'Dumb Cane (Dieffenbachia spp.)', 'Elephant Ear (Alocasia spp.)', 'English Ivy (Hedera helix)', 'Hyacinth (Hyacinthus orientalis)', 'Iron Cross begonia (Begonia masoniana)', 'Jade plant (Crassula ovata)', 'Kalanchoe', 'Lilium (Hemerocallis)', 'Lily of the valley (Convallaria majalis)', 'Money Tree (Pachira aquatica)', 'Monstera Deliciosa (Monstera deliciosa)', 'Orchid', 'Parlor Palm (Chamaedorea elegans)', 'Peac

In [ ]:
# @title Plots

plt.figure(figsize=(15, 12))
images, labels = next(train_ds)
for i in range(min(9, len(images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(list(class_names)[labels[i].astype(int)])
    plt.axis("off")
plt.show()

In [ ]:
# @title Data Augmentation
data_augmentation = tf.keras.Sequential(
  [
    layers.RandomFlip("horizontal", input_shape=(IMG_SIZE, IMG_SIZE,3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
plt.figure(figsize=(12, 10))
images, labels = next(train_ds)
for i in range(min(9, len(images))):
    augmented_images = data_augmentation(images)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_images[0].numpy())
    plt.axis("off")
plt.show()

In [ ]:
# @title Model
def make_model():
    base_model = ResNet50(weights="imagenet", include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))

    for layer in base_model.layers[:-10]:
        layer.trainable = False

    model = tf.keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.2),
        layers.Dense(128, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(1024, activation='relu'),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=['accuracy']
    )

    return model

In [ ]:
# @title Callbacks

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir,
                                                      histogram_freq=1)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="model_checkpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    mode='max')

callbacks = [tensorboard_callback,
             checkpoint_callback,
             early_stopping_callback]

In [ ]:
# @title Training
model = make_model()

history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=EPOCHS,
                    callbacks=callbacks)

Epoch 1/15


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0383 - loss: 3.7747

In [ ]:
%tensorboard --logdir logs/fit